# Step 3a · Features (part 1): wind, rain, land, population and satellite embeddings

**Purpose**: attach explanatory variables to every "county × event" row of the Step 2c label table. Step 3 is split into two notebooks:

- **3a**: wind, rain, land cover, terrain, population and socio-economics, users of electricity-dependent medical equipment, AlphaEarth satellite embeddings
- **3b**: Sentinel-1 flood extent and night lights. These two are computationally heavy and need the list of "counties affected by the hurricane" that 3a produces

**Structure**

| Part | Content | Data source | Needs Earth Engine | Approx. time |
| --- | --- | --- | --- | --- |
| A | Sample: county × event rows of the 9 events | Step 2c results | No | 1 minute |
| B | County locations and areas | TIGER county boundaries | Yes | 1 minute |
| C | Hurricane track: distance to the track, inside the wind radii or not | IBTrACS best track | No | 2 minutes |
| D | Analyzed gusts and wind speed | NOAA RTMA (2.5 km, hourly) | Yes | 5–15 minutes |
| E | Rainfall | PRISM (about 4 km, daily) | Yes | 3–10 minutes |
| F | Land cover and terrain | NLCD 2019, SRTM | Yes | 3–10 minutes |
| G | Population and socio-economics | Census ACS 5-year estimates | No | 1 minute |
| H | Users of electricity-dependent medical equipment | HHS emPOWER | No | 1 minute |
| I | A second set of customer totals, and the outage labels recomputed with it (for the robustness checks of Step 4) | Step 2c and 2d results, ACS housing units | No | 1 minute |
| J | Merge into the feature table; check that features and outages line up | — | No | 1 minute |
| K | AlphaEarth satellite embeddings | Google Satellite Embedding (annual) | Yes | 10–30 minutes |

Every part saves its result to the `us_flood_outage/step3a_cache` folder, so a rerun after an interruption skips what is already done.

**Requirements**

1. An Earth Engine Cloud project, set as `PROJECT_ID` in cell 2
2. A Census API key, read from a Colab secret named `CENSUS_KEY` (or set in cell 2 for a local run); the key is not stored in the notebook

In [ ]:
# 1. Environment, output folder, chart style
import os, time
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, TMP = "/content/drive/MyDrive/us_flood_outage", "/content"
except ImportError:                     # in local Jupyter: use the current folder
    OUT = "us_flood_outage" if os.path.isdir("us_flood_outage") else "../us_flood_outage"    # next to the notebook, or one level up
    TMP = f"{OUT}/tmp"
CACHE = f"{OUT}/step3a_cache"
for d in (OUT, TMP, CACHE):
    os.makedirs(d, exist_ok=True)
print("Output folder:", OUT)

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

def spearman(a, b):
    # Spearman rank correlation: rank first, then take the ordinary correlation
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank()) if ok.sum() >= 5 else np.nan

def cached(name, compute):
    # read the result from the cache folder if it is there, otherwise compute and save it
    path = f"{CACHE}/{name}.csv"
    if not os.path.exists(path):
        compute().to_csv(path, index=False)
    return pd.read_csv(path, dtype={"fips": str})

STATE_ABBR = {"01": "AL", "12": "FL", "13": "GA", "22": "LA", "28": "MS", "37": "NC", "45": "SC", "48": "TX"}
C_POWER = "#2a78d6"
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## Parameters

The definition of "affected by the hurricane" is fixed before looking at the results: a county counts as exposed in an event if it was **ever inside the 34-knot wind radius**, or received **at least 100 mm of rain in 3 consecutive days** (`exposed`). 34 knots (about 63 km/h) is the lower limit of tropical-storm-force wind.

In [ ]:
# 2. Parameters
PROJECT_ID = "ee-yourname"       # Earth Engine Cloud project ID
CENSUS_KEY = ""                  # empty = read the secret named CENSUS_KEY from Colab Secrets (recommended)

TRACK_DAYS = (-3, 7)             # use the track from 3 days before landfall to 7 days after
WIND_DAYS  = (-2, 5)             # analyzed wind: maximum from 2 days before landfall to 5 days after
RAIN_DAYS  = (-2, 7)             # rainfall: 2 days before landfall to 7 days after
SWATH_KT   = 34                  # a county inside the wind radius of this strength counts as exposed
RAIN_3D_MM = 100                 # or with at least this much rain (mm) in 3 consecutive days

EE_BATCH   = 120                 # counties per Earth Engine request
EMB_BATCH  = 60                  # counties per request for the embeddings (many bands, so smaller batches)
EMB_SCALE  = 1000                # grid size (m) at which the embeddings are averaged
EMB_YEAR   = {2017: 2017, 2018: 2017, 2020: 2019, 2021: 2020, 2022: 2021, 2024: 2023}   # event year → embedding year (the year before the event; nothing earlier exists for 2017)
ACS_YEAR   = {2017: 2019, 2018: 2019, 2020: 2019, 2021: 2019, 2022: 2022, 2024: 2022}   # event year → which ACS 5-year vintage

OUT_THRESH, CALM_HOURS, MIN_PEAK = 0.10, 24, 100     # definition of the outage labels, the same as in Step 2c; leave unchanged
TC_TOL     = 0.10                # EAGLE-I's own customer totals are used in a state only if their state sum is within this share of the statewide total
TC_RANGE   = (0.5, 2.0)          # and only for counties whose figure is within this multiple of the housing-unit estimate

## A. Sample

Read the Step 2c label table and keep only the 9 events that passed the screening, and the states with data (`state_ok`). Each row is the outage label of one county in one event; the following parts add features to it.

Counties without any outage record (`no_record`) stay in the table with their flag for now and are removed at the modeling stage.

In [ ]:
# 3. Events and sample
EVENTS = pd.DataFrame([
    dict(key="harvey2017",   name="Harvey",   year=2017, landfall="2017-08-26 03:00", atcf="AL092017"),
    dict(key="irma2017",     name="Irma",     year=2017, landfall="2017-09-10 13:00", atcf="AL112017"),
    dict(key="florence2018", name="Florence", year=2018, landfall="2018-09-14 11:15", atcf="AL062018"),
    dict(key="michael2018",  name="Michael",  year=2018, landfall="2018-10-10 17:30", atcf="AL142018"),
    dict(key="laura2020",    name="Laura",    year=2020, landfall="2020-08-27 06:00", atcf="AL132020"),
    dict(key="sally2020",    name="Sally",    year=2020, landfall="2020-09-16 09:45", atcf="AL192020"),
    dict(key="zeta2020",     name="Zeta",     year=2020, landfall="2020-10-28 21:00", atcf="AL282020"),
    dict(key="ida2021",      name="Ida",      year=2021, landfall="2021-08-29 16:55", atcf="AL092021"),
    dict(key="ian2022",      name="Ian",      year=2022, landfall="2022-09-28 19:05", atcf="AL092022"),
    dict(key="beryl2024",    name="Beryl",    year=2024, landfall="2024-07-08 08:40", atcf="AL022024"),
    dict(key="helene2024",   name="Helene",   year=2024, landfall="2024-09-27 03:10", atcf="AL092024"),
]).set_index("key")
EVENTS["landfall"] = pd.to_datetime(EVENTS["landfall"])

screen = pd.read_csv(f"{OUT}/step2c_event_screening.csv", index_col="event")
usable = [k for k in EVENTS.index if k in screen.index and bool(screen.loc[k, "usable"])]
EVENTS = EVENTS.loc[usable]

keep = ["event", "fips", "name", "state", "year", "customers_est", "customers_used", "peak_customers_out", "peak_time", "peak_pct",
        "hours_above", "hours_rel", "days_to_restore", "affected", "censored", "censored_rel", "denom_raised", "record_share", "no_record",
        "s1_first_day"]
labels = pd.read_csv(f"{OUT}/step2c_labels_county_event.csv", dtype={"fips": str}, parse_dates=["peak_time"])
sample = labels[labels["event"].isin(usable) & labels["state_ok"]][keep].reset_index(drop=True)
sample["long_outage"] = sample["affected"] & (sample["hours_above"] >= 24)        # outage of more than 24 hours
FIPS_ALL = sorted(sample["fips"].unique())
def event_fips(key):
    return sorted(sample.loc[sample["event"] == key, "fips"])

print(f"{len(usable)} events, {len(sample):,} rows (county × event), {len(FIPS_ALL)} counties; {int(sample['affected'].sum()):,} affected rows, {int(sample['long_outage'].sum()):,} with outages over 24 hours")
sample.groupby("event", sort=False).agg(counties=("fips", "size"), affected=("affected", "sum"), long_outage=("long_outage", "sum"),
                                        no_record=("no_record", "sum"))

## B. Connect to Earth Engine; county locations and areas

A county's location is the "internal point" that the Census assigns to it (a representative point guaranteed to lie inside the county); the area counts land only.

This cell also defines `county_reduce`, a function used repeatedly below: it takes the mean (or maximum) of a raster within each county, fetches the result in batches and retries automatically after a failure.

In [ ]:
# 4. [EE] Sign in to Earth Engine; county internal points and land areas
import ee
ee.Authenticate()                         # follow the link to authorize
ee.Initialize(project=PROJECT_ID)
counties_fc = ee.FeatureCollection("TIGER/2018/Counties").filter(ee.Filter.inList("STATEFP", list(STATE_ABBR)))

def ee_rows(fc, props, tries=3):
    # fetch a few properties of a feature collection (Earth Engine drops features where any of them is empty)
    for attempt in range(tries):
        try:
            return fc.reduceColumns(ee.Reducer.toList(len(props)), props).get("list").getInfo()
        except Exception as err:
            if attempt == tries - 1:
                raise
            print(f"  request failed, retrying in {20 * (attempt + 1)} s:", str(err)[:150])
            time.sleep(20 * (attempt + 1))

def county_reduce(img, bands, fips, scale, how="mean", batch=None):
    # for each band of the image, the mean (how="mean") or maximum (how="max") within each county
    reducer = ee.Reducer.mean() if how == "mean" else ee.Reducer.max()
    if len(bands) == 1:
        reducer = reducer.setOutputs(bands)                      # with a single band, name the result after the band rather than "mean"
    batch, rows = batch or EE_BATCH, []
    for i in range(0, len(fips), batch):
        part = counties_fc.filter(ee.Filter.inList("GEOID", fips[i:i + batch]))
        stats = img.select(bands).reduceRegions(collection=part, reducer=reducer, scale=scale, tileScale=4)
        rows += ee_rows(stats, ["GEOID"] + bands)
    return pd.DataFrame(rows, columns=["fips"] + bands).set_index("fips").reindex(fips).reset_index()

def county_points():
    g = pd.DataFrame(ee_rows(counties_fc, ["GEOID", "INTPTLAT", "INTPTLON", "ALAND"]), columns=["fips", "lat", "lon", "aland"])
    g["lat"], g["lon"] = g["lat"].astype(float), g["lon"].astype(float)
    g["land_km2"] = g.pop("aland").astype(float) / 1e6
    return g

geo = cached("county_points", county_points).set_index("fips").reindex(FIPS_ALL)
print(f"{geo['lat'].notna().sum()} / {len(FIPS_ALL)} counties have a location and an area")
tidy(geo.head(3), 2)

## C. Hurricane track: distance to the track, inside the wind radii or not

This uses the IBTrACS best track (the track as revised afterwards by the U.S. National Hurricane Center). Every 3 hours it gives the position of the hurricane's center, the maximum sustained wind, and the radii of 34-, 50- and 64-knot winds in the northeast, southeast, southwest and northwest quadrants. The track is first interpolated to 15 minutes; then for every county:

| Column | Meaning |
| --- | --- |
| `track_dist_km` | Shortest distance from the county's internal point to the track |
| `track_vmax_kt` | Maximum sustained wind (knots) when the hurricane was closest to the county |
| `closest_hours` | Hours after landfall at the moment of closest approach (negative = before landfall) |
| `right_of_track` | The county lies to the right of the direction of motion. In the Northern Hemisphere the right side of a hurricane usually has the stronger wind |
| `wind_swath_kt` | The strongest wind radius the county was ever inside: 0 (never inside the 34-knot radius), 34, 50 or 64 |

IBTrACS in Earth Engine ends in May 2024 and lacks Beryl and Helene, so the complete North Atlantic file (about 60 MB) is downloaded from NOAA directly.

A wind radius is "the farthest extent" in each quadrant, and wind is not uniform inside it, so `wind_swath_kt` errs on the generous side. The analyzed gusts of part D complement it.

In [ ]:
# 5. Download IBTrACS, extract each hurricane's track and interpolate it to 15 minutes
IBTRACS_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/"
               "v04r01/access/csv/ibtracs.NA.list.v04r01.csv")
QUADS = ["NE", "SE", "SW", "NW"]
RADII = [f"USA_R{kt}_{q}" for kt in (34, 50, 64) for q in QUADS]

def download(url, path, label="", min_bytes=1000):
    # download a file to path; skip it if it already exists
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        return path
    print(f"Downloading {label or url} …")
    with requests.get(url, stream=True, timeout=180, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        with open(path + ".part", "wb") as f:
            for block in r.iter_content(chunk_size=1 << 22):
                f.write(block)
    if os.path.getsize(path + ".part") < min_bytes:
        os.remove(path + ".part")
        raise RuntimeError(f"The downloaded file is too small; the site may have blocked the request. Open {url} in a browser, download it manually, place it in the us_flood_outage folder under the same name, then rerun this cell.")
    os.replace(path + ".part", path)
    return path

def read_tracks():
    name = IBTRACS_URL.rsplit("/", 1)[1]
    path = f"{OUT}/{name}" if os.path.exists(f"{OUT}/{name}") else download(IBTRACS_URL, f"{TMP}/{name}", "IBTrACS", 5e6)
    need = ["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "TRACK_TYPE", "USA_ATCF_ID", "USA_WIND"] + RADII
    # row 2 holds the units and is skipped; keep_default_na=False keeps the basin code "NA" (North Atlantic) from being read as missing
    raw = pd.read_csv(path, skiprows=[1], keep_default_na=False, low_memory=False, usecols=lambda c: c in need)
    raw = raw[raw["USA_ATCF_ID"].isin(EVENTS["atcf"])]
    if "TRACK_TYPE" in raw:
        raw = raw[~raw["TRACK_TYPE"].str.lower().str.contains("spur")]      # drop spur segments, keep the main track
    out = pd.DataFrame({"event": raw["USA_ATCF_ID"].map({a: k for k, a in EVENTS["atcf"].items()}),
                        "time": pd.to_datetime(raw["ISO_TIME"])})
    for src, dst in [("LAT", "lat"), ("LON", "lon"), ("USA_WIND", "wind_kt")] + [(c, c[4:].lower()) for c in RADII]:
        out[dst] = pd.to_numeric(raw[src], errors="coerce")
    return out.dropna(subset=["lat", "lon"]).sort_values(["event", "time"])

track_pts = cached("tracks", read_tracks)
track_pts["time"] = pd.to_datetime(track_pts["time"])
missing = [k for k in EVENTS.index if k not in set(track_pts["event"])]
if missing:
    raise RuntimeError(f"No track found in the IBTrACS file for: {missing}")

tracks = {}
for key, ev in EVENTS.iterrows():
    t = track_pts[track_pts["event"] == key].drop_duplicates("time").set_index("time").drop(columns="event")
    radii = [c for c in t.columns if c.startswith("r")]
    t[radii] = t[radii].fillna(0)                                # no radius given = no wind of that strength
    t = t.resample("15min").interpolate("time")
    day0 = ev["landfall"].normalize()
    tracks[key] = t.loc[day0 + pd.Timedelta(days=TRACK_DAYS[0]): day0 + pd.Timedelta(days=TRACK_DAYS[1] + 1)]
pd.DataFrame({k: {"points": len(t), "first": t.index.min(), "last": t.index.max(), "max_wind_kt": t["wind_kt"].max(),
                  "max_r34_nmi": t[[c for c in t.columns if c.startswith("r34")]].max().max()} for k, t in tracks.items()}).T

In [ ]:
# 6. Each county's distance to the track and the strongest wind radius it was inside
NMI_KM = 1.852

def distance_km(lat1, lon1, lat2, lon2):
    # great-circle distance between two points (km)
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(lon2 - lon1)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    # bearing of point 2 seen from point 1 (degrees, 0 = north, clockwise)
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(lon2 - lon1)
    y = np.sin(dl) * np.cos(p2)
    x = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return np.degrees(np.arctan2(y, x)) % 360

rows = []
for key, ev in EVENTS.iterrows():
    t, fips = tracks[key], event_fips(key)
    clat, clon = geo.loc[fips, "lat"].values[None, :], geo.loc[fips, "lon"].values[None, :]
    tlat, tlon = t["lat"].values[:, None], t["lon"].values[:, None]
    dist = distance_km(tlat, tlon, clat, clon)                                  # rows = time steps, columns = counties
    quad = (np.nan_to_num(bearing_deg(tlat, tlon, clat, clon)) // 90).astype(int).clip(0, 3)   # 0 NE, 1 SE, 2 SW, 3 NW
    swath = np.zeros(len(fips), dtype=int)
    for kt in (34, 50, 64):
        radius = t[[f"r{kt}_{q.lower()}" for q in QUADS]].values * NMI_KM        # radius in the four quadrants at each time step (km)
        reach = np.take_along_axis(radius, quad, axis=1)                        # the radius in the quadrant each county is in
        swath[((dist <= reach) & (reach > 0)).any(axis=0)] = kt
    i = np.where(np.isnan(dist), np.inf, dist).argmin(axis=0)                   # the time step closest to each county
    before, after = np.clip(i - 4, 0, len(t) - 1), np.clip(i + 4, 0, len(t) - 1)
    heading = bearing_deg(t["lat"].values[before], t["lon"].values[before], t["lat"].values[after], t["lon"].values[after])
    to_county = bearing_deg(t["lat"].values[i], t["lon"].values[i], clat[0], clon[0])
    rows.append(pd.DataFrame({
        "event": key, "fips": fips,
        "track_dist_km": dist[i, np.arange(len(fips))],
        "track_vmax_kt": t["wind_kt"].values[i],
        "closest_hours": (t.index[i] - ev["landfall"]) / pd.Timedelta(hours=1),
        "right_of_track": ((to_county - heading) % 360) < 180,
        "wind_swath_kt": swath,
    }))
track_feat = pd.concat(rows, ignore_index=True)
track_feat.to_csv(f"{CACHE}/track_features.csv", index=False)
pd.crosstab(track_feat["event"], track_feat["wind_swath_kt"]).reindex(list(EVENTS.index))

## D. Analyzed gusts and wind speed (RTMA)

RTMA is NOAA's hourly surface weather analysis, which blends station, radar and satellite data onto a 2.5 km grid. For each event, the maximum gust and maximum wind speed of every grid cell from 2 days before landfall to 5 days after are taken first and then summarized within the county:

| Column | Meaning |
| --- | --- |
| `gust_mean_ms` | Mean over the county's grid cells of their maximum gust (m/s): the gust the county experienced generally |
| `gust_peak_ms` | The highest of the grid cells' maximum gusts within the county |
| `wind_mean_ms` | Mean over the county's grid cells of their maximum sustained wind |

**Known limitations**: stations near a hurricane's core are often destroyed or lose power, so RTMA underestimates wind there. Thunderstorm gusts unrelated to the hurricane within those 7 days are counted as well, so RTMA suits the role of a continuous explanatory variable but not that of deciding "was this county affected by this hurricane" — the wind radii of part C do that.

In [ ]:
# 7. [EE] RTMA: maximum gust and maximum wind for each event and county
rtma = ee.ImageCollection("NOAA/NWS/RTMA")

def day_str(key, offset):
    return (EVENTS.loc[key, "landfall"].normalize() + pd.Timedelta(days=offset)).strftime("%Y-%m-%d")

def rtma_event(key):
    fips = event_fips(key)
    hours = (rtma.filterDate(day_str(key, WIND_DAYS[0]), day_str(key, WIND_DAYS[1] + 1))
                 .filter(ee.Filter.listContains("system:band_names", "GUST")))       # skip the occasional hour without a gust band
    peak = hours.select(["GUST", "WIND"]).max()
    mean = county_reduce(peak, ["GUST", "WIND"], fips, 2500, "mean").rename(columns={"GUST": "gust_mean_ms", "WIND": "wind_mean_ms"})
    top  = county_reduce(peak, ["GUST", "WIND"], fips, 2500, "max").rename(columns={"GUST": "gust_peak_ms", "WIND": "wind_peak_ms"})
    return mean.merge(top, on="fips").assign(event=key)

parts = []
for key in EVENTS.index:
    parts.append(cached(f"rtma_{key}", lambda: rtma_event(key)))
    print(f"{key}: {parts[-1]['gust_mean_ms'].notna().sum()} / {len(parts[-1])} counties with a value")
rtma_feat = pd.concat(parts, ignore_index=True)
tidy(rtma_feat.groupby("event", sort=False)[["gust_mean_ms", "gust_peak_ms", "wind_mean_ms"]].agg(["median", "max"]))

## E. Rainfall (PRISM)

PRISM is Oregon State University's daily rainfall grid (about 4 km), which blends rain gauges and radar and is a standard rainfall dataset for the conterminous United States. For each event, the 10 days from 2 days before landfall to 7 days after are taken; the mean rainfall of each county is computed for each day and then summarized:

| Column | Meaning |
| --- | --- |
| `rain_total_mm` | Total rainfall over the 10 days (mm) |
| `rain_max1d_mm` | The wettest single day |
| `rain_max3d_mm` | The wettest 3 consecutive days |

A PRISM "day" runs from 12:00 UTC of the previous day to 12:00 UTC of the day itself, half a day off the calendar day; this matters little for the 3-day and 10-day totals.

In [ ]:
# 8. [EE] PRISM: daily rainfall for each event and county
prism = ee.ImageCollection("OREGONSTATE/PRISM/ANd").select("ppt")
N_RAIN = RAIN_DAYS[1] - RAIN_DAYS[0] + 1
DAY_BANDS = [f"d{i:02d}" for i in range(N_RAIN)]

def prism_event(key):
    # one band per day; if a day has images of two versions, they are averaged
    daily = ee.Image.cat([prism.filterDate(day_str(key, RAIN_DAYS[0] + i), day_str(key, RAIN_DAYS[0] + i + 1)).mean().rename(b)
                          for i, b in enumerate(DAY_BANDS)])
    return county_reduce(daily, DAY_BANDS, event_fips(key), 4000, "mean").assign(event=key)

parts = []
for key in EVENTS.index:
    parts.append(cached(f"prism_{key}", lambda: prism_event(key)))
rain_daily = pd.concat(parts, ignore_index=True)
days = rain_daily[DAY_BANDS]
rain_feat = rain_daily[["event", "fips"]].assign(
    rain_total_mm=days.sum(axis=1, min_count=N_RAIN),
    rain_max1d_mm=days.max(axis=1),
    rain_max3d_mm=days.T.rolling(3).sum().max().values)
tidy(rain_feat.groupby("event", sort=False)[["rain_total_mm", "rain_max1d_mm", "rain_max3d_mm"]].agg(["median", "max"]))

## F. Land cover and terrain

These are county features that do not vary by event, all computed over land only (open water inside the county boundary is left out):

| Column | Meaning | Why it may matter |
| --- | --- | --- |
| `forest_pct` | Forest as a share of land area | Trees falling on lines are the main cause of wind outages |
| `developed_pct` | Developed share | Degree of urbanization |
| `wetland_pct` | Wetland share | Low and flood-prone; hard for repair crews to reach |
| `farmland_pct` | Cropland and pasture share | — |
| `impervious_pct` | Mean share of impervious surface | Urban flooding |
| `low_lying_pct` | Share of land below 5 m elevation | Storm surge and coastal flooding |
| `elevation_m` | Mean elevation | — |

Land cover comes from the single NLCD 2019 epoch and is shared by all events (land cover changes slowly). It is sampled at 300 m spacing, which is enough for county-level shares.

In [ ]:
# 9. [EE] NLCD land cover and SRTM terrain
nlcd = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").filter(ee.Filter.eq("system:index", "2019")).first()
lc   = nlcd.select("landcover")
land = lc.neq(11)                                                # 11 = open water
elev = ee.Image("USGS/SRTMGL1_003").select("elevation")
STATIC_BANDS = ["forest_pct", "developed_pct", "wetland_pct", "farmland_pct", "impervious_pct", "low_lying_pct", "elevation_m"]
static_img = ee.Image.cat([
    lc.gte(41).And(lc.lte(43)).multiply(100),                    # 41–43 forest
    lc.gte(21).And(lc.lte(24)).multiply(100),                    # 21–24 developed
    lc.eq(90).Or(lc.eq(95)).multiply(100),                       # 90, 95 wetlands
    lc.eq(81).Or(lc.eq(82)).multiply(100),                       # 81, 82 pasture and cropland
    nlcd.select("impervious"),
    elev.lt(5).multiply(100),
    elev,
]).rename(STATIC_BANDS).updateMask(land)

static_feat = cached("static", lambda: county_reduce(static_img, STATIC_BANDS, FIPS_ALL, 300, "mean"))
print(f"{static_feat['forest_pct'].notna().sum()} / {len(FIPS_ALL)} counties with a value")
tidy(static_feat[STATIC_BANDS].describe().loc[["min", "50%", "max"]])

## G. Population and socio-economics (Census ACS)

This uses the ACS 5-year estimates: the 2015–2019 vintage for the 2017–2021 events, and the 2018–2022 vintage for the 2022 and 2024 events.

| Column | Meaning |
| --- | --- |
| `pop`, `pop_density` | Population; population per km² of land |
| `median_income` | Median household income (US dollars) |
| `poverty_pct` | Share of people below the poverty line |
| `age65_pct` | Share of people aged 65 and over |
| `renter_pct` | Share of renter households |
| `mobile_home_pct` | Mobile homes as a share of housing. Mobile homes withstand wind poorly |
| `no_vehicle_pct` | Share of households without a vehicle |
| `electric_heat_pct` | Share of households heating with electricity |
| `median_year_built` | Median year the housing was built |
| `vacant_pct`, `seasonal_pct` | Share of vacant housing; share of housing for seasonal or recreational use |
| `housing_units` | Total housing units |

Part I uses `housing_units` for a second estimate of the customer total: in counties with many holiday homes and in counties with few residents, the estimate from population is too low.

In [ ]:
# 10. Census ACS: population and socio-economics
AGE65 = [f"B01001_{i:03d}E" for i in list(range(20, 26)) + list(range(44, 50))]      # the age groups 65 and over, men and women
ACS_VARS = ["B01003_001E", "B19013_001E", "B17001_001E", "B17001_002E", "B25001_001E", "B25003_001E", "B25003_003E",
            "B25024_001E", "B25024_010E", "B25035_001E", "B08201_001E", "B08201_002E", "B25040_001E", "B25040_004E",
            "B25002_001E", "B25002_003E", "B25004_006E", "B01001_001E"] + AGE65

def census_key():
    if CENSUS_KEY:
        return CENSUS_KEY
    try:
        from google.colab import userdata
        return userdata.get("CENSUS_KEY")
    except Exception:
        return ""

def acs_raw(year):
    key = census_key()
    if not key:
        raise RuntimeError("No Census API key found. Add CENSUS_KEY in Colab Secrets and switch on Notebook access, or put it in cell 2 temporarily.")
    parts = []
    for s in STATE_ABBR:
        r = requests.get(f"https://api.census.gov/data/{year}/acs/acs5", timeout=120,
                         params={"get": ",".join(ACS_VARS), "for": "county:*", "in": f"state:{s}", "key": key})
        if r.status_code != 200 or not r.text.lstrip().startswith("["):
            raise RuntimeError(f"The Census API returned no data ({year}, state {s}, status {r.status_code}): {r.text[:200]}")
        data = r.json()
        parts.append(pd.DataFrame(data[1:], columns=data[0]))
    raw = pd.concat(parts, ignore_index=True)
    raw.insert(0, "fips", raw.pop("state") + raw.pop("county"))
    return raw

def acs_features(year):
    raw = cached(f"acs_{year}", lambda: acs_raw(year)).set_index("fips")
    v = raw[ACS_VARS].apply(pd.to_numeric, errors="coerce")
    v = v.where(v >= 0)                                          # the Census uses large negative numbers for "no value"
    pct = lambda a, b: v[a] / v[b] * 100
    return pd.DataFrame({
        "acs_year": year, "pop": v["B01003_001E"], "median_income": v["B19013_001E"],
        "poverty_pct": pct("B17001_002E", "B17001_001E"), "age65_pct": v[AGE65].sum(axis=1) / v["B01001_001E"] * 100,
        "renter_pct": pct("B25003_003E", "B25003_001E"), "mobile_home_pct": pct("B25024_010E", "B25024_001E"),
        "no_vehicle_pct": pct("B08201_002E", "B08201_001E"), "electric_heat_pct": pct("B25040_004E", "B25040_001E"),
        "median_year_built": v["B25035_001E"], "vacant_pct": pct("B25002_003E", "B25002_001E"),
        "seasonal_pct": pct("B25004_006E", "B25002_001E"), "housing_units": v["B25001_001E"],
    }).reset_index()

acs_feat = pd.concat([acs_features(y) for y in sorted(set(ACS_YEAR.values()))], ignore_index=True)
acs_feat = acs_feat[acs_feat["fips"].isin(FIPS_ALL)]
print(f"ACS: counties per vintage {acs_feat.groupby('acs_year')['fips'].nunique().to_dict()}")
tidy(acs_feat.drop(columns=["fips", "acs_year"]).describe().loc[["min", "50%", "max"]])

## H. Users of electricity-dependent medical equipment (HHS emPOWER)

The emPOWER data of the U.S. Department of Health and Human Services give, for each county, the number of Medicare beneficiaries who use electricity-dependent medical equipment at home (ventilators, oxygen concentrators, power wheelchairs and so on). These people are most at risk in a long outage.

| Column | Meaning |
| --- | --- |
| `medicare_benes` | Number of Medicare beneficiaries |
| `dme_users` | Of whom users of electricity-dependent medical equipment |
| `dme_per_1000` | Such users per 1,000 residents |

**Two limitations**: the service only offers the latest figures, not those of the event year; and to protect privacy, small counts of 1–10 are shown as 11. So it can only approximate "how large this group has been in the county over the long run".

In [ ]:
# 11. HHS emPOWER: users of electricity-dependent medical equipment
EMPOWER_URL = ("https://services2.arcgis.com/ZQ4jTQn6k7VPXEwO/arcgis/rest/services/"
               "HHS_emPOWER_REST_Service_Public/FeatureServer/2/query")

def empower_raw():
    parts = []
    for abbr in STATE_ABBR.values():
        r = requests.get(EMPOWER_URL, timeout=120, params={
            "where": f"State='{abbr}'", "outFields": "FIPS_Code,Medicare_Benes,Power_Dependent_Devices_DME",
            "returnGeometry": "false", "f": "json"})
        data = r.json()
        if "features" not in data:
            raise RuntimeError(f"emPOWER returned no data ({abbr}): {str(data)[:200]}")
        parts.append(pd.DataFrame([f["attributes"] for f in data["features"]]))
    raw = pd.concat(parts, ignore_index=True)
    return pd.DataFrame({"fips": raw["FIPS_Code"].astype(str).str.zfill(5), "medicare_benes": raw["Medicare_Benes"],
                         "dme_users": raw["Power_Dependent_Devices_DME"]})

empower = cached("empower", empower_raw)
empower = empower[empower["fips"].isin(FIPS_ALL)].drop_duplicates("fips")
print(f"emPOWER: {len(empower)} / {len(FIPS_ALL)} counties with a value; {int(empower['dme_users'].sum()):,} users of electricity-dependent medical equipment in total")

## I. A second set of customer totals, and the outage labels recomputed with it

**Why**: the main outcome `hours_above` divides by each county's customer total, and that total is an estimate (county population × statewide customers per person). Step 2d found two things:

- In counties with few residents this estimate is too low, because the number of meters follows the number of homes, farms and shops, not only the resident population
- The county customer totals that come with the 2024 EAGLE-I file add up to roughly the statewide total in 5 states and are clearly too high in the other 3, so they cannot replace the estimate everywhere

So the main denominator stays as it is, and this part prepares a **second denominator** and recomputes the labels with it. Step 4 fits the model once with each set of labels to see whether the conclusions agree.

**Where the second denominator comes from**

1. `customers_hu`: the county's housing units (ACS) × statewide customers per housing unit (statewide customers ÷ statewide housing units)
2. `total_customers_2024`: the customer total that comes with the 2024 EAGLE-I file (the Step 2d result)
3. `customers_alt`: where the EAGLE-I figures of a state add up to within 10% of the statewide total, and the county's figure is between 0.5 and 2 times `customers_hu`, the EAGLE-I figure is used (`alt_source` is `eaglei`); otherwise `customers_hu` (`alt_source` is `housing`)
4. `customers_alt_used`: as in Step 2c, if the cleaned outage peak is larger than the denominator, the denominator is raised to the peak

**The recomputed labels**

| Column | Meaning |
| --- | --- |
| `peak_pct_alt`, `affected_alt` | Peak outage share and the affected flag with the second denominator (same definitions as Step 2c) |
| `hours_above_1h` | `hours_above` recomputed with the main denominator from the hourly curves. Only for comparison with the next column |
| `hours_above_alt` | Outage duration with the second denominator, from the hourly curves |
| `censored_alt` | Not restored within the observation window, with the second denominator |

**Notes**

- The Step 2c labels were computed from 15-minute data; this part uses the hourly curves that Step 2c saved. So the labels are first recomputed hourly with the main denominator (`hours_above_1h`) to confirm that they agree with the original `hours_above`, and only then are the two denominators compared
- Hourly curves were saved only for counties affected under the main definition. For the other counties, `hours_above_alt` is 0 if the peak stays below 10% with the second denominator; if it rises above 10%, there is no curve to compute from and the value is left empty. The number of such counties is printed
- The EAGLE-I customer totals are those of 2024, so they are slightly too high for earlier events; no adjustment is made here

In [ ]:
# 12. A second set of customer totals, and the outage labels recomputed with it
alt = sample[["event", "fips", "state", "year", "customers_est", "customers_used", "peak_customers_out", "peak_time", "affected", "hours_above"]].copy()
peak = alt["peak_customers_out"].fillna(0)

# (1) estimate from housing units: county housing units × statewide customers per housing unit
cov = pd.read_csv(f"{OUT}/step2c_coverage_by_state.csv")
state_total = cov[cov["year"] == cov["year"].max()].set_index("state")["total_customers"]
hu_ref = acs_features(max(ACS_YEAR.values()))                                   # all counties of the 8 states
per_hu = state_total / hu_ref.groupby(hu_ref["fips"].str[:2].map(STATE_ABBR))["housing_units"].sum()
hu = acs_feat.set_index(["fips", "acs_year"])["housing_units"]
alt["customers_hu"] = hu.reindex(pd.MultiIndex.from_arrays([alt["fips"], alt["year"].map(ACS_YEAR)])).values * alt["state"].map(per_hu).values

# (2) EAGLE-I's own customer totals: used only in states where their sum agrees with the statewide total
tc_path = f"{OUT}/step2d_total_customers_2024_by_county.csv"
if os.path.exists(tc_path):
    tc = pd.read_csv(tc_path, dtype={"fips": str}).set_index("fips")["year_max"].dropna()
else:
    tc = pd.Series(dtype=float)
    print("Step 2d customer totals not found; the second denominator uses the housing-unit estimate everywhere")
tc_state = pd.Series({abbr: tc[[f for f in tc.index if f[:2] == s]].sum() for s, abbr in STATE_ABBR.items()})
tc_tab = pd.DataFrame({"eaglei_total": tc_state, "state_total": state_total.reindex(tc_state.index)})
tc_tab["ratio"] = tc_tab["eaglei_total"] / tc_tab["state_total"]
tc_tab["used"] = (tc_tab["ratio"] - 1).abs() <= TC_TOL
alt["total_customers_2024"] = alt["fips"].map(tc)
in_range = (alt["total_customers_2024"] / alt["customers_hu"]).between(*TC_RANGE)
use_tc = alt["state"].isin(tc_tab.index[tc_tab["used"]]) & in_range
alt["alt_source"] = np.where(use_tc, "eaglei", "housing")
alt["customers_alt"] = np.where(use_tc, alt["total_customers_2024"], alt["customers_hu"])
alt["customers_alt_used"] = np.maximum(alt["customers_alt"], peak)                 # raise the denominator to the peak where the peak is larger
alt["peak_pct_alt"] = peak / alt["customers_alt_used"] * 100
alt["affected_alt"] = (alt["peak_pct_alt"] >= OUT_THRESH * 100) & (peak >= MIN_PEAK)

# (3) recompute the outage duration from the hourly curves: with the main denominator first, then with the second
def hours_1h(wide, den, peak_hour):
    # the same rule as Step 2c: hours at or above 10%, counted up to the first 24 consecutive hours below 10% after the peak
    idx = wide.index
    share = wide.div(den, axis=1).clip(upper=1.0)
    after_peak = pd.DataFrame(idx.values[:, None] >= peak_hour.values[None, :], index=idx, columns=wide.columns)
    calm = (share < OUT_THRESH).astype(float)[::-1].rolling(CALM_HOURS, min_periods=CALM_HOURS).min()[::-1].eq(1) & after_peak
    restored = calm.idxmax().where(calm.any())
    end = idx[-1] + pd.Timedelta(hours=1)
    until = pd.DataFrame(idx.values[:, None] < restored.fillna(end).values[None, :], index=idx, columns=wide.columns)
    return ((share >= OUT_THRESH) & until).sum().astype(float), restored.isna()

hourly = pd.read_csv(f"{OUT}/step2c_panel_hourly.csv.gz", dtype={"fips": str}, parse_dates=["time"])
parts = []
for key in usable:
    rows = alt[alt["event"] == key].set_index("fips")
    wide = hourly[hourly["event"] == key].pivot(index="time", columns="fips", values="customers_out").sort_index()
    wide = wide[[f for f in wide.columns if f in rows.index]]
    if wide.shape[1] == 0:
        continue
    r = rows.loc[wide.columns]
    peak_hour = r["peak_time"].dt.floor("h")
    h_main, _ = hours_1h(wide, r["customers_used"], peak_hour)
    h_alt, open_alt = hours_1h(wide, r["customers_alt_used"], peak_hour)
    parts.append(pd.DataFrame({"event": key, "fips": list(wide.columns), "hours_above_1h": h_main.values, "hours_above_alt": h_alt.values,
                               "open_alt": open_alt.values}))
curves = pd.concat(parts, ignore_index=True)
alt = alt.merge(curves, on=["event", "fips"], how="left")
has_curve = alt["hours_above_1h"].notna()
still_zero = ~has_curve & (alt["peak_pct_alt"] < OUT_THRESH * 100) & (alt["hours_above"] == 0)   # no curve, and below 10% with the second denominator as well
alt.loc[still_zero, "hours_above_alt"] = 0.0
alt["hours_above_alt"] = alt["hours_above_alt"].where(alt["customers_alt_used"].notna())          # empty where there is no second denominator
alt["censored_alt"] = alt["affected_alt"] & alt.pop("open_alt").eq(True)
ALT_COLS = ["customers_hu", "total_customers_2024", "alt_source", "customers_alt", "customers_alt_used", "peak_pct_alt", "affected_alt",
            "hours_above_1h", "hours_above_alt", "censored_alt"]

# (4) comparison
A = alt[alt["affected"]].copy()
A["size_class"] = pd.cut(A["customers_est"], [0, 5000, 20000, 100000, np.inf], labels=["<5k", "5k-20k", "20k-100k", ">100k"])
def den_summary(g):
    return pd.Series({
        "affected_rows": len(g),
        "peak_over_pop_pct": (g["peak_customers_out"] > g["customers_est"]).mean() * 100,
        "peak_over_hu_pct":  (g["peak_customers_out"] > g["customers_hu"]).mean() * 100,
        "peak_over_alt_pct": (g["peak_customers_out"] > g["customers_alt"]).mean() * 100,
        "alt_vs_pop_median": (g["customers_alt"] / g["customers_est"]).median(),
        "still_affected_pct": g["affected_alt"].mean() * 100,
        "median_hours_main": g["hours_above_1h"].median(), "median_hours_alt": g["hours_above_alt"].median(),
        "over_24h_main": (g["hours_above_1h"] >= 24).sum(), "over_24h_alt": (g["hours_above_alt"] >= 24).sum(),
        "over_7d_main": (g["hours_above_1h"] >= 168).sum(), "over_7d_alt": (g["hours_above_alt"] >= 168).sum(),
    })
den_check = pd.DataFrame({**{name: den_summary(g) for name, g in A.groupby("size_class", observed=True)}, "all": den_summary(A)}).T
den_check.index.name = "customers_est"
den_check.to_csv(f"{OUT}/step3a_denominator_check.csv")

gap = (A["hours_above_1h"] - A["hours_above"]).abs()
rho_1h, rho_alt = spearman(A["hours_above_1h"], A["hours_above"]), spearman(A["hours_above_alt"], A["hours_above_1h"])
newly = alt["affected_alt"] & ~alt["affected"]
print(f"Source of the second denominator (county × event rows): {alt['alt_source'].value_counts().to_dict()}; affected rows raised to the peak: main {int((A['peak_customers_out'] > A['customers_est']).sum())}, second {int((A['peak_customers_out'] > A['customers_alt']).sum())}")
print(f"Hourly recomputation against the 15-minute labels (main denominator, {len(A):,} affected rows): median difference {gap.median():.2f} hours, largest {gap.max():.2f} hours, Spearman {rho_1h:.3f}")
print(f"Outage duration with the two denominators: Spearman {rho_alt:.3f}; {int((~A['affected_alt']).sum())} rows no longer affected with the second denominator, {int(newly.sum())} newly affected (of which {int((newly & alt['hours_above_alt'].isna()).sum())} have no curve, so the duration is left empty)")
display(tidy(tc_tab, 2))
tidy(den_check, 2)

## J. Merge into the feature table; check that features and outages line up

The parts above are merged onto the sample to give `step3a_features_county_event.csv`: one row per "county × event", with the Step 2c labels and the labels recomputed in part I first, and the features after them.

Then `exposed` (affected by this hurricane) is flagged using the definition fixed in the parameters, and three checks follow:

1. **Table**: in each event, how many of the affected counties lie within the exposed area. A county with an outage of more than 24 hours that is not exposed either had an outage unrelated to the hurricane, or the exposed area is drawn too narrowly
2. **Figure 1**: within each event, whether counties with stronger gusts had longer outages
3. **Figure 2**: whether the share of counties with outages over 24 hours rises with the strength of the wind radius they were inside

In [ ]:
# 13. Merge the feature table and flag the exposed counties
feat = (sample.merge(alt[["event", "fips"] + ALT_COLS], on=["event", "fips"], how="left")
              .merge(track_feat, on=["event", "fips"], how="left")
              .merge(rtma_feat.drop(columns="wind_peak_ms"), on=["event", "fips"], how="left")
              .merge(rain_feat, on=["event", "fips"], how="left")
              .merge(static_feat, on="fips", how="left")
              .merge(geo[["lat", "lon", "land_km2"]].reset_index(), on="fips", how="left"))
feat["acs_year"] = feat["year"].map(ACS_YEAR)
feat = feat.merge(acs_feat, on=["fips", "acs_year"], how="left").merge(empower, on="fips", how="left")
feat["pop_density"]  = feat["pop"] / feat["land_km2"]
feat["dme_per_1000"] = feat["dme_users"] / feat["pop"] * 1000
feat["exposed"] = (feat["wind_swath_kt"] >= SWATH_KT) | (feat["rain_max3d_mm"] >= RAIN_3D_MM)
feat.to_csv(f"{OUT}/step3a_features_county_event.csv", index=False)

HAZARD = ["track_dist_km", "wind_swath_kt", "gust_mean_ms", "gust_peak_ms", "wind_mean_ms", "rain_total_mm", "rain_max1d_mm", "rain_max3d_mm"]
PLACE  = STATIC_BANDS + ["pop", "pop_density", "median_income", "poverty_pct", "age65_pct", "renter_pct", "mobile_home_pct", "no_vehicle_pct",
                         "electric_heat_pct", "median_year_built", "vacant_pct", "seasonal_pct", "dme_per_1000"]
print(f"Feature table: {len(feat):,} rows, {len(HAZARD)} hazard features, {len(PLACE)} county features")

exposure = pd.DataFrame({k: {
    "counties": len(g), "exposed": int(g["exposed"].sum()),
    "in_wind_swath": int((g["wind_swath_kt"] >= SWATH_KT).sum()), "rain_only": int((g["exposed"] & (g["wind_swath_kt"] < SWATH_KT)).sum()),
    "affected": int(g["affected"].sum()), "affected_exposed_pct": g.loc[g["affected"], "exposed"].mean() * 100,
    "long_outage": int(g["long_outage"].sum()), "long_outage_exposed_pct": g.loc[g["long_outage"], "exposed"].mean() * 100,
    "long_outage_not_exposed": int((g["long_outage"] & ~g["exposed"]).sum()),
    "exposed_no_record": int((g["exposed"] & g["no_record"]).sum()),
    "rho_gust_hours": spearman(g.loc[g["exposed"], "gust_mean_ms"], g.loc[g["exposed"], "hours_above"]),
    "rho_rain_hours": spearman(g.loc[g["exposed"], "rain_max3d_mm"], g.loc[g["exposed"], "hours_above"]),
} for k, g in feat.groupby("event", sort=False)}).T
exposure.to_csv(f"{OUT}/step3a_exposure_by_event.csv")
tidy(exposure, 2)

In [ ]:
# 14. Figure 1: gusts against outage duration within each event (exposed counties only)
keys = list(EVENTS.index)
ncol = 3
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(12, 3.1 * nrow), squeeze=False)
for ax, key in zip(axes.flat, keys):
    g = feat[(feat["event"] == key) & feat["exposed"] & ~feat["no_record"]]
    ax.scatter(g["gust_mean_ms"], g["hours_above"] / 24, s=14, color=C_POWER, edgecolor=SURFACE, linewidth=0.4, alpha=0.7)
    ax.set_title(f"{EVENTS.loc[key, 'name']} {EVENTS.loc[key, 'year']}")
    ax.text(1, 1.03, f"n = {len(g)}, Spearman {spearman(g['gust_mean_ms'], g['hours_above']):.2f}", transform=ax.transAxes,
            ha="right", color=MUTED, fontsize=8.5)
    ax.set_ylim(bottom=0)
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("Mean of maximum gust in the county (m/s)")
for ax in axes[:, 0]:
    ax.set_ylabel("Days with 10%+ out")
fig.suptitle("Gusts and outage duration, exposed counties", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step3a_fig1_gust_vs_duration.png"); plt.show()

In [ ]:
# 15. Figure 2: does the share of counties with long outages rise with the wind radius they were inside
ok = feat[~feat["no_record"]]
by_swath = ok.groupby("wind_swath_kt").agg(rows=("fips", "size"), long_outage_pct=("long_outage", lambda s: s.mean() * 100),
                                           median_hours=("hours_above", "median")).reindex([0, 34, 50, 64])
names = ["Outside 34 kt", "34–49 kt", "50–63 kt", "64 kt or more"]
fig, ax = plt.subplots(figsize=(7.2, 4))
ax.bar(names, by_swath["long_outage_pct"], color=C_POWER, width=0.62)
for x, (pct, n) in enumerate(zip(by_swath["long_outage_pct"], by_swath["rows"])):
    if pd.notna(pct):
        ax.text(x, pct + 1.5, f"{pct:.0f}%", ha="center", color=INK, fontsize=10)
        ax.text(x, -0.13, f"n = {int(n):,}", ha="center", va="top", color=MUTED, fontsize=9, transform=ax.get_xaxis_transform())
ax.set_ylim(0, 105); ax.grid(axis="x", visible=False)
ax.set_ylabel("Share of county-events (%)")
ax.set_title("Outages of more than 24 hours, by strongest wind radius reached")
fig.savefig(f"{OUT}/step3a_fig2_outage_by_wind_radius.png"); plt.show()
tidy(by_swath)

## K. AlphaEarth satellite embeddings

Google's AlphaEarth Foundations model compresses a full year of several kinds of satellite imagery into one 64-dimensional vector (an "embedding") per 10 m pixel. It is not a hand-defined indicator but surface characteristics the model learned by itself; differences between trees, buildings, cropland and water are all in there.

Here it is summarized by county:

- `e00`–`e63`: mean embedding over the county's land
- `b00`–`b63`: mean embedding weighted by built-up area, that is, what "the places with houses and power lines" look like

So that the features do not "see" the aftermath, the embedding of the year **before** the event is used (for the 2017 events only 2017 itself is available, as nothing earlier exists). The averaged vectors are rescaled to length 1.

Step 4 will compare whether adding these 64 dimensions improves the prediction of outage duration. The last cell checks the extraction: the first principal component of the embeddings should correlate clearly with the forest share from part F.

In [ ]:
# 16. [EE] Satellite embeddings: for each county and each year needed
emb_col = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
A = [f"A{i:02d}" for i in range(64)]
E, B = [f"e{i:02d}" for i in range(64)], [f"b{i:02d}" for i in range(64)]
built_frac = (ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S").filterDate("2015-01-01", "2016-01-01").first()
                .select("built_surface").divide(10000))          # built-up share of each 100 m cell

sample["emb_year"] = sample["year"].map(EMB_YEAR)
def emb_year(year):
    fips = sorted(sample.loc[sample["emb_year"] == year, "fips"].unique())
    img = emb_col.filterDate(f"{year}-01-01", f"{year + 1}-01-01").mosaic().select(A)
    whole = county_reduce(img.updateMask(land).rename(E), E, fips, EMB_SCALE, "mean", batch=EMB_BATCH)
    # the mask value acts as a weight: the larger the built-up share, the larger the weight
    built = county_reduce(img.updateMask(built_frac).rename(B), B, fips, EMB_SCALE, "mean", batch=EMB_BATCH)
    return whole.merge(built, on="fips").assign(year=year)

parts = []
for year in sorted(set(EMB_YEAR[y] for y in EVENTS["year"])):
    parts.append(cached(f"embeddings_{year}", lambda: emb_year(year)))
    print(f"{year}: {parts[-1]['e00'].notna().sum()} / {len(parts[-1])} counties with a value")
emb = pd.concat(parts, ignore_index=True)
for cols in (E, B):                                              # rescale the averaged vectors to length 1
    emb[cols] = emb[cols].div(np.sqrt((emb[cols] ** 2).sum(axis=1, min_count=64)), axis=0)
emb[["fips", "year"] + E + B].to_csv(f"{OUT}/step3a_embeddings_county_year.csv", index=False)
print(f"Saved {len(emb):,} rows (county × year) × {len(E) + len(B)} dimensions")

In [ ]:
# 17. Figure 3: first principal component of the embeddings against forest share
latest = emb.sort_values("year").drop_duplicates("fips", keep="last").dropna(subset=E).set_index("fips")
X = latest[E].values - latest[E].values.mean(axis=0)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
explained = S ** 2 / (S ** 2).sum()
pc = pd.DataFrame(U[:, :3] * S[:3], index=latest.index, columns=["pc1", "pc2", "pc3"]).join(static_feat.set_index("fips")[STATIC_BANDS])
rho = {c: max((abs(spearman(pc[p], pc[c])), p) for p in ["pc1", "pc2", "pc3"]) for c in ["forest_pct", "developed_pct", "farmland_pct", "wetland_pct"]}
best = rho["forest_pct"][1]
fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.scatter(pc["forest_pct"], pc[best], s=12, color=C_POWER, edgecolor=SURFACE, linewidth=0.4, alpha=0.7)
ax.set_xlabel("Forest share of land (%)"); ax.set_ylabel(f"Embedding {best.upper()}")
ax.set_title("Do the embeddings see forest?", pad=22)
ax.text(0, 1.03, f"{len(pc)} counties; |Spearman| = {rho['forest_pct'][0]:.2f}; first 3 components hold {explained[:3].sum():.0%} of the variance",
        transform=ax.transAxes, color=MUTED, fontsize=9)
fig.savefig(f"{OUT}/step3a_fig3_embedding_check.png"); plt.show()
print("Share of variance explained by the first 10 components:", np.round(explained[:10], 3))
print("Largest correlation (absolute) between each land-cover share and the first 3 components:", {c: round(float(v[0]), 2) for c, v in rho.items()})

## Checklist

The thresholds were set before looking at the results. The last item is an exception: the comparison of the two denominators was computed beforehand from the Step 2c and 2d results, and is listed here to confirm that this run gives the same answer.

In [ ]:
# 18. Step 3a checklist
def verdict(passed):
    return "pass" if passed else "fail"

ok = feat[~feat["no_record"]]
have_hazard = feat[HAZARD].notna().all(axis=1).mean() * 100
have_place  = feat[PLACE].notna().all(axis=1).mean() * 100
long_exposed = ok.loc[ok["long_outage"], "exposed"].mean() * 100
steps = by_swath["long_outage_pct"].dropna().values
rho_gust = spearman(ok.loc[ok["exposed"], "gust_mean_ms"], ok.loc[ok["exposed"], "hours_above"])
have_emb = sample.merge(emb[["fips", "year", "e00"]], left_on=["fips", "emb_year"], right_on=["fips", "year"], how="left")["e00"].notna().mean() * 100

checks = [
    ("Rows with all hazard features (%)", round(have_hazard, 1), "≥ 99", have_hazard >= 99),
    ("Rows with all county features (%)", round(have_place, 1), "≥ 98", have_place >= 98),
    ("Counties with outages over 24 hours that are exposed (%)", round(long_exposed, 1), "≥ 80", long_exposed >= 80),
    ("Share of long outages rises with the wind radius", " < ".join(f"{s:.0f}%" for s in steps), "rising at every step", bool(len(steps) >= 3 and (np.diff(steps) > 0).all())),
    ("Spearman correlation of gust and outage duration among exposed counties", round(float(rho_gust), 2), "≥ 0.3", rho_gust >= 0.3),
    ("Rows with satellite embeddings (%)", round(have_emb, 1), "≥ 99", have_emb >= 99),
    ("Correlation (absolute) of an embedding component with forest share", round(float(rho["forest_pct"][0]), 2), "≥ 0.5", rho["forest_pct"][0] >= 0.5),
    ("Spearman correlation of outage duration between the two sets of customer totals", round(float(rho_alt), 2), "≥ 0.9", rho_alt >= 0.9),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step3a_checklist.csv", index=False)
check_tab

## Output files

All written to the `us_flood_outage` folder:

- `step3a_features_county_event.csv`, `step3a_exposure_by_event.csv`, `step3a_denominator_check.csv`, `step3a_checklist.csv`
- `step3a_embeddings_county_year.csv`
- `step3a_fig1_gust_vs_duration.png`, `step3a_fig2_outage_by_wind_radius.png`, `step3a_fig3_embedding_check.png`